In [ ]:
!pip install roboflow ultralytics sahi

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="mz3cNkxiO8av9JAjZbS3")
project = rf.workspace("my-ws-lwkgs").project("tl_detector-coivv")
version = project.version(19)
dataset = version.download("coco")

In [ ]:
from sahi.slicing import slice_coco
from sahi.utils.file import save_json
from sahi.utils.coco import Coco
import os

out_dir = f"{dataset.location}/sliced"
try:
    os.rmdir(out_dir)
except:
    pass

coco_dict, coco_path = slice_coco(
    coco_annotation_file_path=f"{dataset.location}/train/_annotations.coco.json",
    image_dir=f"{dataset.location}/train",
    output_coco_annotation_file_name="annotations",
    output_dir=f"{out_dir}/images",
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    min_area_ratio=0.1,  # Add this
    ignore_negative_samples=False,
)

coco = Coco.from_coco_dict_or_path(coco_dict, image_dir=f"{out_dir}/images")
result = coco.split_coco_as_train_val(train_split_rate=0.85)

In [ ]:
from sahi.utils.coco import Coco, export_coco_as_yolo

sliced_yolo_dir=f"{out_dir}/yolo_dataset"
try:
    os.rmdir(sliced_yolo_dir)
except:
    pass

data_yml_path = export_coco_as_yolo(
    output_dir=sliced_yolo_dir,
    train_coco=result["train_coco"],
    val_coco=result["val_coco"]
)

In [ ]:
from ultralytics import YOLO

#sliced_yolo_dir="/workspace/tl_detector-19/sliced/yolo_dataset"

model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{sliced_yolo_dir}/data.yml",
    epochs=100,
    imgsz=320,
    rect=True,
    multi_scale=False,
    batch=256,
    workers=32,
    name='tl_detector',
    augment=True,  # включаем ручной контроль над аугментацией
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    auto_augment='none',
    erasing=0.0
)

In [ ]:
print(f"Result: {results.save_dir}/confusion_matrix_normalized.png")
print(f"Model: {results.save_dir}/weights/best.pt")
print(f"Calibration images: {out_dir}/images")